# Phase 2: how much of the 99% is real?

Runs the Phase 2 checks from `src/phase2.py` on every dataset, then zips `results/` for download.

| Experiment | Question |
|---|---|
| `protocols` | Study's order vs our Phase 1 vs a fully fair order, repeated over `SEEDS` random splits, with significance tests |
| `official` | NSL-KDD: learn on KDDTrain+, test on KDDTest+ (new attack types) |
| `holdout` | Hide one attack family while learning; is it still caught? |
| `shortcut` | Remove the most important columns; does the model still work? |
| `cross` | Learn on CIC-IDS2017, test on CSE-CIC-IDS2018, and the reverse |

**Setup:** Internet ON; attach the raw-data dataset (`ris-m26-full`). GPU is only needed if `DEEP = True`.
Cleaning is re-run here because Phase 2 needs two extra bookkeeping columns (attack name, source file).
If a session runs out of time, shorten `DATASETS` and run the rest in a second session.

In [ ]:
DATASETS = ["nsl_kdd", "unsw_nb15", "cse_cic_ids2018", "cic_ids2017"]
EXPERIMENTS = "protocols,official,holdout,shortcut"   # "official" is skipped for all but NSL-KDD
SEEDS = 5          # repeated random splits; seed 42 is the Phase 1 split
DEEP = False       # True adds FT-Transformer + stacked FT team to the fair runs (needs GPU, slow)
CROSS = True       # needs cic_ids2017 and cse_cic_ids2018 in DATASETS

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", "phase-2", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"

RAW_MARKERS = {
    "nsl_kdd": "nsl_kdd/KDDTrain+*",
    "unsw_nb15": "unsw_nb15/UNSW_NB15_training-set.csv",
    "cic_ids2017": "cic*2017/*WorkingHours*.csv",
    "cse_cic_ids2018": "c*2018/*-2018.csv",
}

def point_at_raw(datasets):
    roots, missing = set(), []
    for ds in datasets:
        hits = glob.glob(f"/kaggle/input/**/{RAW_MARKERS[ds]}", recursive=True)
        if hits:
            roots |= {os.path.dirname(os.path.dirname(h)) for h in hits}
        else:
            missing.append(ds)
    if missing or len(roots) != 1:
        for root, _, files in os.walk("/kaggle/input"):
            for fn in files:
                print("  ", os.path.join(root, fn))
        raise AssertionError(f"raw data not found for {missing}, or found in several places: {roots}")
    os.environ["RIS_DATA_RAW"] = roots.pop()
    print("raw data folder:", os.environ["RIS_DATA_RAW"])

point_at_raw(DATASETS)

def run_live(cmd):
    """Run a pipeline script and stream its output as it happens."""
    p = subprocess.Popen(cmd, cwd=SRC, env=os.environ, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

In [ ]:
!pip install -q imbalanced-learn catboost lightgbm tabulate

In [ ]:
# Cleaning (same as Phase 1, plus the attack-name and source-file columns)
for ds in DATASETS:
    print(f"=== {ds}: preprocess ===", flush=True)
    assert run_live([sys.executable, "-u", "preprocess.py", "--dataset", ds]) == 0, ds

In [ ]:
for ds in DATASETS:
    print(f"\n=== {ds}: phase 2 ===", flush=True)
    cmd = [sys.executable, "-u", "phase2.py", "--dataset", ds, "--experiments", EXPERIMENTS, "--seeds", str(SEEDS)]
    if DEEP:
        cmd.append("--deep")
    assert run_live(cmd) == 0, ds

In [ ]:
# Check: the "phase1" order at seed 42 must repeat our Phase 1 NSL-KDD numbers
import pandas as pd
path = f"{WORK}/results/nsl_kdd/phase2/protocols.csv"
if os.path.exists(path):
    p = pd.read_csv(path)
    print(p[(p.Protocol == "phase1") & (p.Seed == 42)].set_index("Model")["Accuracy"].round(7))
    print("Phase 1 was: Random Forest 0.9955823, XGBoost 0.9961007, hybrid 0.9960331")

In [ ]:
if CROSS:
    assert run_live([sys.executable, "-u", "phase2.py", "--cross"]) == 0

In [ ]:
assert run_live([sys.executable, "-u", "phase2_report.py"]) == 0

for ds in DATASETS:
    path = f"{WORK}/results/{ds}/phase2/protocols_summary.csv"
    if os.path.exists(path):
        s = pd.read_csv(path)
        print(f"\n{ds}")
        print(s[["Protocol", "Model", "Accuracy_mean", "Accuracy_std", "FPR_mean"]].round(4).to_string(index=False))

shutil.make_archive(f"{WORK}/phase2_results", "zip", f"{WORK}/results")
print("\nDownload phase2_results.zip from the Output panel.")